In [5]:
import pandas as pd



In [6]:

train = pd.read_csv('../data/val_with_predictions.csv')

In [7]:
train['predicted_state'] = train['emotional_state']
train['predicted_intensity'] = train['intensity']


In [9]:
def decide_action(state, intensity, stress, energy, time_of_day):
    # Priority 1: High Stress/Anxiety
    if stress >= 4 or state in ['overwhelmed', 'anxious']:
        if intensity >= 4:
            return "box_breathing", "now"
        return "grounding", "within_15_min"
    
    # Priority 2: Low Energy
    if energy <= 2:
        if time_of_day == 'night':
            return "rest", "tonight"
        return "rest", "later_today"
    
    # Priority 3: Productive States
    if state == 'focused' or state == 'calm':
        if energy >= 4:
            return "deep_work", "now"
        return "light_planning", "tomorrow_morning" if time_of_day == 'night' else "later_today"
    
    # Fallback for Mixed/Restless
    if state == 'mixed':
        return "journaling", "within_15_min"
    
    return "pause", "later_today"

In [10]:
# This replaces the old loop and the old assignment cell
results = train.apply(lambda row: decide_action(
    row['predicted_state'], 
    row['predicted_intensity'], 
    row['stress_level'], 
    row['energy_level'], 
    row['time_of_day']
), axis=1)

# Split the results into two columns
train['what_to_do'] = [r[0] for r in results]
train['when_to_do'] = [r[1] for r in results]

# Show the result
train[['predicted_state', 'what_to_do', 'when_to_do']].head()

,predicted_state,what_to_do,when_to_do
0,neutral,pause,later_today
1,overwhelmed,grounding,within_15_min
2,calm,light_planning,later_today
3,restless,box_breathing,now
4,focused,light_planning,later_today


In [11]:

print("Columns in current dataframe:", train.columns.tolist())

unsure_cases = train[train['uncertain_flag'] == 1]
print(f"\nThere are {len(unsure_cases)} cases where the model is uncertain.")
unsure_cases[['journal_text', 'predicted_state', 'confidence']].head()

Columns in current dataframe: ['id', 'journal_text', 'ambience_type', 'duration_min', 'sleep_hours', 'energy_level', 'stress_level', 'time_of_day', 'previous_day_mood', 'face_emotion_hint', 'reflection_quality', 'emotional_state', 'intensity', 'clean_text', 'predicted_state', 'predicted_intensity', 'confidence', 'uncertain_flag', 'what_to_do', 'when_to_do']

There are 170 cases where the model is uncertain.


,journal_text,predicted_state,confidence
0,"lowkey felt pretty even, but this was better t...",neutral,0.326192
1,"for a while i was more tired than i expected, ...",overwhelmed,0.381269
3,Honestly I felt distracted which surprised me.,restless,0.454898
5,"lowkey felt locked in for a bit, but mountain ...",focused,0.252289
7,woke up feeling less tense. i had to restart o...,calm,0.342696


In [13]:
# The assignment requires these specific columns:
# id, predicted_state, predicted_intensity, confidence, uncertain_flag, what_to_do, when_to_do

final_export = train[[
    'id', 
    'predicted_state', 
    'predicted_intensity', 
    'confidence', 
    'uncertain_flag', 
    'what_to_do', 
    'when_to_do'
]]

# Save the final deliverable to the data folder
final_export.to_csv('../data/predictions.csv', index=False)

print("🚀 SUCCESS: predictions.csv has been created in the /data/ folder!")

🚀 SUCCESS: predictions.csv has been created in the /data/ folder!
